# OmniConnector and disaggregated execution

## Learning goals
- Understand what crosses stage boundaries
- Compare shared-memory and network transfer
- Separate stage disaggregation from model parallelism

> **Mac track:** executable cells use lightweight simulations. Boxes labelled **Linux GPU lab** show how the same concept maps to the official runtime.

## Connector responsibility

A connector answers **how** data moves, while the stage processor defines **what** the next stage needs.

Payloads may include token IDs, hidden states, embeddings, KV caches, codec codes, latents, or metadata.

In [ ]:
from omni_tutorial.pipeline import Connector
c=Connector('shared-memory'); payload={'kind':'hidden_states','shape':(1,128,4096)}
received=c.transfer(payload)
print(received,'transfer count',c.transfers)

## Placement tradeoff

```text
Same process:      simplest, no isolation
Shared memory:     efficient single-node transfer
Network/RDMA:      flexible multi-node placement, added transfer cost
```

A boundary is worthwhile when independent scaling/placement benefits exceed orchestration and communication overhead.

In [ ]:
payload_mb=[1,64,512]
for mb in payload_mb:
    for gbps in [10,100]:
        ms=mb*8/gbps
        print(f"{mb:3} MB over {gbps:3} Gbps: idealized {ms:5.1f} ms")

## Key distinction

- **Across stages:** disaggregation and replicas.
- **Within one stage:** tensor, pipeline, data, expert, sequence, or CFG parallelism.